In [ ]:
import os
import glob
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
from collections import defaultdict

import sunpy.visualization.colormaps as cm
from astropy.io import fits 
from skimage import io

from skimage import exposure
from skimage.transform import resize   
from matplotlib.path import Path

import sunpy.map
from sunpy.map.header_helper import make_heliographic_header
from skimage.draw import rectangle

import matplotlib.dates as mdates
import datetime
import string

import numpy as np
import cv2

##### Function Mean and Standard Deviation of Each Map

In [ ]:
def process_aia_image(file, cmap, channel, year, timestamp):
    aia_map = sunpy.map.Map(file)
    shape = (1440, 2880)
    carr_header = make_heliographic_header(aia_map.date,
                                               aia_map.observer_coordinate,
                                               shape, frame='stonyhurst',)
    outmap = aia_map.reproject_to(carr_header)
    stonyhurst_reprojection = outmap.data
    flip = stonyhurst_reprojection[::-1,:]

    nan_mask = np.isnan(flip)
    
    center_row = flip.shape[0] // 2

    center_non_nan_cols = np.where(~nan_mask[center_row])[0]
    
    if len(center_non_nan_cols) > 0:
        start_col = center_non_nan_cols[0]
        end_col = center_non_nan_cols[-1] + 1
    
        # Find the first and last non-NaN in the columns of the bounding box
        rows_with_nan_in_box = np.any(~nan_mask[:, start_col:end_col], axis=1)
        start_row = np.argmax(rows_with_nan_in_box)
        end_row = len(rows_with_nan_in_box) - np.argmax(rows_with_nan_in_box[::-1])
    
        cropped_image = flip[start_row:end_row, start_col:end_col]
    else:
        cropped_image = np.array([])  
    
    nan_mask = np.isnan(cropped_image)
    cropped_image[nan_mask] = np.interp(np.flatnonzero(nan_mask), np.flatnonzero(~nan_mask), cropped_image[~nan_mask])

    # new_shape = 256
    # if cropped_image.shape[0] > 0:
    #     resized_image = cv2.resize(cropped_image, (new_shape, new_shape), interpolation=cv2.INTER_LINEAR)
    # else:
    #     resized_image = np.full((new_shape, new_shape), np.nan)

    log_i = np.log(cropped_image)   
    # out_path_npy = f"../DATASET/log_scale_npy/{channel}/{year}/{timestamp}"
    # os.makedirs(os.path.dirname(out_path_npy), exist_ok=True)
    # np.save(out_path_npy, log_i)
    
    # Remove NAN Effects
    non_nan_log_i = log_i[~np.isnan(log_i)]
    stdd = np.std(non_nan_log_i)
    meann = np.mean(non_nan_log_i)
    
    # std_log_image = ((log_i - meann)/stdd)

    # plt.imshow(std_log_image, cmap=cmap, origin='lower')
    # out_path = f"../DATASET/save_maps/{channel}/{year}/{timestamp}"
    # os.makedirs(os.path.dirname(out_path), exist_ok=True)
    # plt.savefig(out_path)
    # plt.close()
    
    return meann, stdd


In [ ]:
records = []
years = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020]
for year in years:
    path_193 = f"../DATASET/AIA/193/{year}/*.*"
    path_171 = f"../DATASET/AIA/171/{year}/*.*"
    path_304 = f"../DATASET/AIA/304/{year}/*.*"

    files_193 = sorted(glob.glob(path_193))
    files_171 = sorted(glob.glob(path_171))
    files_304 = sorted(glob.glob(path_304))

    # Build dicts
    aia_dict_193 = {os.path.basename(f)[3:11]: f for f in files_193}
    aia_dict_171 = {os.path.basename(f)[3:11]: f for f in files_171}
    aia_dict_304 = {os.path.basename(f)[3:11]: f for f in files_304}

    common_timestamps = sorted(
        set(aia_dict_193.keys()) &
        set(aia_dict_171.keys()) &
        set(aia_dict_304.keys())
    )

    print(f"\nYear {year}: Found {len(common_timestamps)} matched dates")

    for i, ts in enumerate(common_timestamps, 1):
        # if year == 2012 and i < 340:
        #     continue  # Skip first 9 images of 2012
        aia_file_193 = aia_dict_193[ts]
        aia_file_171 = aia_dict_171[ts]
        aia_file_304 = aia_dict_304[ts]

        print(f"({i:03d}) {ts} — "
              f"AIA 193: {os.path.basename(aia_file_193)} | "
              f"AIA 171: {os.path.basename(aia_file_171)} | "
              f"AIA 304: {os.path.basename(aia_file_304)}")

        mean_193, std_193 = process_aia_image(aia_file_193, 'sdoaia193', '193', year, ts)
        mean_171, std_171 = process_aia_image(aia_file_171, 'sdoaia171', '171', year, ts)
        mean_304, std_304 = process_aia_image(aia_file_304, 'sdoaia304', '304', year, ts)
        
        # append row as dict
        records.append({
            "date": ts,
            "mean_193": mean_193,
            "std_193": std_193,
            "mean_171": mean_171,
            "std_171": std_171,
            "mean_304": mean_304,
            "std_304": std_304
        })

# build dataframe
df = pd.DataFrame(records)

# show or save
print(df.head())
df.to_csv("../DATASET/stonyhurst_aia_statistics.csv", index=False)
